In [ ]:
# 1. Install system dependencies (GPU detection & utilities)
!apt-get update -qq
!apt-get install -y pciutils lshw zstd curl

# 2. Install Ollama and Python helpers
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q huggingface_hub pyngrok

print("✅ Setup Phase 1 Complete: Dependencies installed.")

In [ ]:
import os
import subprocess
import time

# 1. Explicitly point to the Colab GPU libraries
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu'

# 2. Start Ollama in the background
print("🚀 Starting Ollama server with GPU support...")
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Give the server a few seconds to initialize
time.sleep(5)

print("✅ Setup Phase 2 Complete: Server running!")
# Verify the GPU is visible
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from huggingface_hub import hf_hub_download
import subprocess

MODEL_FILE = "Qwen3.5-4B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

print("📥 Downloading model (this takes a minute)...")
model_path = hf_hub_download(
    repo_id="HauhauCS/Qwen3.5-4B-Uncensored-HauhauCS-Aggressive",
    filename=MODEL_FILE,
    local_dir="/content/ollama-model"
)

# Define the correct Modelfile
modelfile_content = f"FROM {model_path}\n" + """
PARAMETER num_ctx 4096
PARAMETER num_gpu 99
PARAMETER temperature 0.7

TEMPLATE \"\"\"{{ if .System }}
<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}
{{ if .Prompt }}
<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}
<|im_start|>assistant
\"\"\"
"""

# Write and build
with open("/content/Modelfile", "w") as f:
    f.write(modelfile_content)

print("🏗️ Building the model in Ollama...")
subprocess.run(["ollama", "create", "qwen3.5-4b", "-f", "/content/Modelfile"])
print("✅ Setup Phase 3 Complete: Model is ready for use!")

In [ ]:
import os
import time
import subprocess
import requests
import json
import gradio as gr

# 1. Kill any broken CPU-bound background processes
subprocess.run(['pkill', '-f', 'ollama'])
time.sleep(2)

# 2. THE FIX: Explicitly point Ollama to the Colab GPU drivers
env = os.environ.copy()
env['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/lib/x86_64-linux-gnu'
env['OLLAMA_HOST'] = '127.0.0.1:11434'

# 3. Start the Ollama server in the background WITH the GPU environment
print("🚀 Booting up the Ollama server (GPU linked)...")
subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Wait for it to fully wake up

# 4. Define the Chat Function
def chat_with_qwen(message, history):
    messages = []
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    try:
        response = requests.post(
            "http://127.0.0.1:11434/api/chat",
            json={
                "model": "qwen3.5-4b",
                "messages": messages,
                "stream": True
            },
            stream=True
        )

        partial_message = ""
        for line in response.iter_lines():
            if line:
                chunk = json.loads(line)
                if 'message' in chunk:
                    content = chunk['message'].get('content', '')
                    partial_message += content
                    yield partial_message

    except Exception as e:
        yield f"❌ Connection Error: {str(e)}"

# 5. Build the UI
print("🎨 Building Chat Interface...")
demo = gr.ChatInterface(
    fn=chat_with_qwen,
    title="🚀 Qwen 3.5 Uncensored (GPU Accelerated)",
    description="Running instantly on T4 GPU via Ollama.",
    theme=gr.themes.Soft()
)

# 6. Launch the UI in the background
print("✅ Server is live! Click the public Gradio link below to chat:")
demo.launch(share=True)

# 7. FORCE THE CELL TO RUN INFINITELY
print("⚙️ Entering infinite loop to keep the cell spinning. Click the stop button to end.")
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Stopped manually. Shutting down gracefully...")
    subprocess.run(['pkill', '-f', 'ollama'])